# Train a U-Net on native-resolution crops (Kaggle GPU)

Import this notebook into Kaggle and press **Save & Run All**. None of the cells asks for input. With the default `MODEL = "r34"` no cell needs editing. To train another run, change only `MODEL` in the settings cell and save the notebook under the name in this table (the evaluation and inference notebooks look for that slug). `r34_v2` and `r34_v2_agree` keep the ResNet-34, its data and its 80 epochs and change the optimization recipe (section 1):

| `MODEL` | Encoder | Save the training notebook as | Checkpoint |
| --- | --- | --- | --- |
| `r34` (default) | `resnet34` | `train-unet-r34-crops-80ep` | `unet_r34_crops.pt` |
| `r50` | `resnet50` | `train-unet-r50-crops` | `unet_r50_crops.pt` |
| `effb3` | `efficientnet-b3` | `train-unet-effb3-crops` | `unet_effb3_crops.pt` |
| `r34_v2` | `resnet34` | `train-unet-r34-v2` | `unet_r34_v2.pt` |
| `r34_v2_agree` | `resnet34` | `train-unet-r34-v2-agree` | `unet_r34_v2_agree.pt` |

**Attach exactly these Kaggle inputs**

1. The competition `filament-segmentation-2026` (Add Input → Competition). It provides the train images and the train JSON.
2. The dataset `ignacioaguirrecandia/solar-code` (the repo zip, extracted as `Solar-Filament-Segmentation-main/`). It has to be a version that contains `filament.crop_config.CROP_MODELS` with the chosen `MODEL`. An older version stops at the imports, or in the settings cell with `unknown crop model`. [VERIFY] re-upload the dataset after the merge.

**Settings**

- Accelerator: **GPU T4**, a single T4 (not "GPU T4 x2").
- Internet: **On**. A cell installs `segmentation-models-pytorch==0.6.0`, and smp downloads the chosen encoder's ImageNet weights from the Hugging Face Hub.
- Save the notebook under the name in the table above. For `r34` that is **`train-unet-r34-crops-80ep`**: the inference notebook (`infer_unet_crops.ipynb`) looks for this notebook's output at `/kaggle/input/notebooks/ignacioaguirrecandia/train-unet-r34-crops-80ep/unet_r34_crops.pt`, which assumes that slug.

**Outputs** (in `/kaggle/working`)

- `unet_r34_crops.pt` (the name in the table): the checkpoint. It is rewritten after every epoch, so a killed session still leaves usable weights.
- `holdout_sweep_<MODEL>.csv`: pooled per-view PQ on the date holdout for each `min_area`. Same columns as `results/postprocess_sweep_2026-11-01.csv`.

This notebook does not write `submission.csv`, does not submit to Kaggle, and does not read the test images.

Expected wall time: for `r34`, about 3 h (80 epochs at about 127 s, measured on 2026-11-02), with the training loop capped at `MAX_TRAIN_HOURS` (5 h). For `r50` and `effb3`, at most about 9 h, with the training loop capped at 8.5 h. For `r34_v2` and `r34_v2_agree`, about 3 to 3.5 h, with the same 5 h cap. The time per epoch of everything but `r34` is an estimate, not a measurement [VERIFY]. The settings cell prints the estimate, and the first epoch prints its real speed and peak GPU memory.


## 1. What this run changes

The reference is the 512 px ResNet-18 run (`notebooks/train_unet.ipynb`, 12 epochs): pooled per-view PQ 0.2886 on the holdout at `min_area` 400 and `close_size` 0, public leaderboard 0.23 (`docs/RESULTS.md`).

| | 512 px ResNet-18 | This run |
| --- | --- | --- |
| Encoder | `resnet18`, ImageNet | `MODEL`: `resnet34` by default, ImageNet |
| Input | whole frame, area-averaged to 512 | random 512 px crops of the native 2048 px frame, no resizing |
| Mask | nearest-downscaled to 512 | native resolution, so thin barbs are not lost to resizing |
| Crop position | not applicable | with probability `P_FG` centered on a random foreground pixel (jitter, clamped inside the frame); otherwise uniform. A frame with no foreground is always uniform |
| Augmentation | none | horizontal flip, vertical flip, 90° rotation (`AUGMENT`) |
| Loss, optimizer | Dice + BCE, Adam 1e-4 | the same |
| Precision | float32 | mixed precision (`torch.amp`) on CUDA, loss in float32 |
| Inference | one 512 px pass, nearest upsample | overlapped 512 px tiles on the native frame, probabilities averaged |

**Recipe v2** (`r34_v2`, `docs/ROADMAP_V2.md` D1). The same ResNet-34, data, Dice + BCE, seed and 80 epochs, with a different optimization recipe: AdamW (weight decay 1e-4), a peak rate of 2e-4 with one warmup epoch and then a cosine toward 0 (its mean rate is about the constant 1e-4 of `r34`), gradient clipping at 1.0, and an exponential moving average of the weights (decay 0.999). The checkpoint stores the average of the last epoch; no epoch is picked on validation. The rate is under 0.2% of its peak during the last 1000 steps, so the average is expected to be very close to the last weights. Any gain belongs to the recipe as a whole: these changes are not separated.

**Agreement weights** (`r34_v2_agree`, D2). `r34_v2` plus a per-pixel weight on the BCE: `0.4 + 0.6 * k / R` on the union, 1 on the background, where `k` of the file's `R` annotator views cover the pixel. The target is still the union and the Dice term is unweighted. The formula is the roadmap's proposal, not a tuned value. The crops are the same as for `r34_v2`.

Everything here is untuned. `EPOCHS`, `CROPS_PER_EPOCH` and `BATCH_SIZE` come from a timing estimate for one T4 that is not recorded in this repo [VERIFY]. The flips and rotations change the apparent orientation relative to the solar disk; filament chirality is not a label here (all four categories are one class), and the effect of the augmentation on the score has not been measured.

The split is exactly `filament.train_data.split_training_images`: the most recent 20% of calendar dates are held out, one row per file name (565 train files and 142 holdout files with 231 annotator views on the current data). The target is the union of every polygon on a file.

**No `DataLoader`.** Each train frame and its union mask are decoded once and kept in RAM as uint8 (about 4.5 GiB for 565 files). A crop is a numpy slice of those arrays plus an optional flip or rotation, and the normalization (divide by 255, repeat to 3 channels, ImageNet mean and standard deviation) runs on the GPU. Batches are therefore built in the main process: worker processes would add copies and complexity. The cost of cropping was not profiled against the GPU step [VERIFY].

**Encoder weights.** smp 0.6.0 pins `resnet34` / `imagenet` to the Hugging Face repo `smp-hub/resnet34.imagenet` at revision `7a57b34f723329ff020b3f8bc41771163c519d0c`. Its Hub license field and its relation to the torchvision `.pth` were not checked [VERIFY]. `resnet50` / `imagenet` is `smp-hub/resnet50.imagenet` at `00cb74e366966d59cd9a35af57e618af9f88efe9`, and `efficientnet-b3` / `imagenet` is `smp-hub/efficientnet-b3.imagenet` at `074c54a6c473e0d294690d49cedb6cf463e7127d`; their licenses were not checked either [VERIFY]. Do not use `ssl`, `swsl` or `advprop` weights.


## 2. `src` on the path

`src` is the directory that contains `filament/crop_training.py`. A checkout in the working directory is used as it is. Otherwise the walk goes through `/kaggle/input` at any depth, skips every `competitions` directory, and takes the shallowest match, for example:

`/kaggle/input/datasets/ignacioaguirrecandia/solar-code/Solar-Filament-Segmentation-main/src`

It is used in place and not copied to `/kaggle/working`, so it does not end up in this notebook's saved output.


In [ ]:
import os
import sys
from pathlib import Path


def _local_src() -> Path | None:
    """A checkout that already has the crop modules (a run off Kaggle)."""
    cwd = Path.cwd().resolve()
    for candidate in (cwd, *cwd.parents):
        src = candidate / "src"
        if (src / "filament" / "crop_training.py").is_file():
            return src
    return None


def _private_dataset_src(root: Path | None = None) -> Path | None:
    """Return the shallowest private-dataset ``src`` under ``root``.

    ``filament/crop_training.py`` may sit at any depth. Directories named
    ``competitions`` are not walked, so the MAGFiLO tree is left alone.
    """
    root = Path("/kaggle/input") if root is None else root
    if not root.is_dir():
        return None
    marker = Path("filament") / "crop_training.py"
    matches: list[Path] = []
    for dirpath, dirnames, _filenames in os.walk(root, topdown=True):
        dirnames[:] = [name for name in dirnames if name != "competitions"]
        current = Path(dirpath)
        if current.name != "src" or not (current / marker).is_file():
            continue
        matches.append(current)
        dirnames.clear()
    if not matches:
        return None
    return min(
        matches,
        key=lambda path: (len(path.relative_to(root).parts), path.as_posix()),
    )


def _put_on_path(src: Path) -> None:
    text = str(src)
    if text not in sys.path:
        sys.path.insert(0, text)
    current = os.environ.get("PYTHONPATH", "")
    parts = [part for part in current.split(os.pathsep) if part]
    if text not in parts:
        os.environ["PYTHONPATH"] = os.pathsep.join([text, *parts]) if parts else text


def _prepare_src() -> Path:
    """Put the repo's ``src`` on ``sys.path`` and return it.

    A checkout in the working directory is used as it is. Otherwise the
    private dataset's ``src`` is used in place. It is not copied to
    ``/kaggle/working``: a copy there would be saved with this notebook's
    output and found again by a later notebook that attaches that output.
    """
    local = _local_src()
    if local is not None:
        _put_on_path(local)
        return local
    source = _private_dataset_src()
    if source is None:
        raise FileNotFoundError(
            "Could not find src/filament/crop_training.py in the working "
            "directory or under /kaggle/input (any depth outside "
            "competitions). Attach the dataset ignacioaguirrecandia/solar-code, "
            "for example "
            "/kaggle/input/datasets/ignacioaguirrecandia/solar-code/"
            "Solar-Filament-Segmentation-main/src. If it is attached, its "
            "version predates the crop modules: upload a new version of the "
            "dataset from the branch that has them. Do not make the GitHub "
            "repository public."
        )
    _put_on_path(source)
    return source


SRC = _prepare_src()
print(f"src={SRC}")
PYTHON = sys.executable

## 3. Install the segmentation library

`segmentation-models-pytorch` version **0.6.0**, MIT license. Kaggle already has PyTorch; this cell does not install it. If the install fails, stop.


In [ ]:
import subprocess

# Pinned. segmentation-models-pytorch 0.6.0 is MIT.
SMP_VERSION = "0.6.0"

completed = subprocess.run(
    [
        PYTHON,
        "-m",
        "pip",
        "install",
        f"segmentation-models-pytorch=={SMP_VERSION}",
    ],
    check=False,
)
if completed.returncode != 0:
    raise RuntimeError(
        f"segmentation-models-pytorch=={SMP_VERSION} did not install. Stop. "
        "Internet has to be On. Kaggle already has PyTorch; this cell only "
        "installs segmentation-models-pytorch (MIT)."
    )

In [ ]:
import segmentation_models_pytorch as smp

print(f"segmentation-models-pytorch {smp.__version__}")
if smp.__version__ != SMP_VERSION:
    raise RuntimeError(
        f"Expected segmentation-models-pytorch {SMP_VERSION}, "
        f"found {smp.__version__}. Stop."
    )

## 4. Imports and settings

`MODEL` is the only setting meant to change. It picks the encoder, `EPOCHS`, `BATCH_SIZE`, `MAX_TRAIN_HOURS` and the optimization recipe (`LEARNING_RATE` and the last block of `CONFIG`) from `filament.crop_config.CROP_MODELS`, and the expected checkpoint from `filament.checkpoint.CROP_CHECKPOINTS`; every other setting is the same for all models. `EPOCHS` is the only number that sets the length of the run. The wall-clock guard `MAX_TRAIN_HOURS` stops the loop before an epoch that would end past it, and the checkpoint then stores the number of **completed** epochs.

The last two blocks are for the holdout evaluation: tile overlap 128 at tile 512 (stride 384, a 5 × 5 grid of 25 tiles on a 2048 px frame), inference batch 8, threshold 0.5, and the `min_area` grid of the 2026-11-01 sweep with `close_size` 0.

The next cell stops with a clear error if there is no CUDA GPU or if the GPU has under 6 GiB.


In [ ]:
import statistics
import time
from pathlib import Path

import torch

from filament import crop_training as ct
from filament.annotations import (
    TRAIN_ANNOTATIONS_PATH,
    TRAIN_IMAGES_DIR,
    load_train_annotations,
)
from filament.checkpoint import checkpoint_fingerprint, crop_checkpoint
from filament.crop_config import (
    DEFAULT_CROP_MODEL,
    CropTrainConfig,
    crop_model_config,
    estimated_train_hours,
)
from filament.crop_data import load_train_cache
from filament.sweep import count_views
from filament.train_data import split_training_images

In [ ]:
# ---- Which model: a name of filament.crop_config.CROP_MODELS ----------------
MODEL = "r34"
PRESET = crop_model_config(MODEL)
EXPECTED = crop_checkpoint(MODEL)

# ---- Training settings (untuned) -------------------------------------------
EPOCHS = PRESET.epochs
CROPS_PER_EPOCH = 4520  # 8 random crops for each of the 565 train files
CROP_SIZE = 512
BATCH_SIZE = PRESET.batch_size
LEARNING_RATE = PRESET.learning_rate
P_FG = 0.67
AUGMENT = True
SEED = 0
USE_AMP = True
MAX_TRAIN_HOURS = PRESET.max_train_hours

# ---- Holdout evaluation: tiled inference at the native resolution ----------
TILE_SIZE = CROP_SIZE
TILE_OVERLAP = 128
INFER_BATCH_SIZE = 8
THRESHOLD = 0.5
MIN_AREAS = (80, 140, 240, 400, 700, 1200, 2000, 3000)
CLOSE_SIZE = 0

# ---- Paths (found, not typed) ----------------------------------------------
IMAGE_DIR = Path(TRAIN_IMAGES_DIR)
ANNOTATIONS_PATH = Path(TRAIN_ANNOTATIONS_PATH)
OUTPUT_DIR = Path("/kaggle/working")
WEIGHTS_PATH = OUTPUT_DIR / Path(EXPECTED.weights_path).name
SWEEP_PATH = OUTPUT_DIR / f"holdout_sweep_{MODEL}.csv"

CONFIG = CropTrainConfig(
    encoder_name=PRESET.encoder_name,
    encoder_weights=PRESET.encoder_weights,
    epochs=EPOCHS,
    crops_per_epoch=CROPS_PER_EPOCH,
    crop_size=CROP_SIZE,
    batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    p_fg=P_FG,
    augment=AUGMENT,
    seed=SEED,
    use_amp=USE_AMP,
    max_train_hours=MAX_TRAIN_HOURS,
    optimizer=PRESET.optimizer,
    weight_decay=PRESET.weight_decay,
    lr_schedule=PRESET.lr_schedule,
    warmup_epochs=PRESET.warmup_epochs,
    grad_clip_norm=PRESET.grad_clip_norm,
    ema_decay=PRESET.ema_decay,
    agreement_weights=PRESET.agreement_weights,
)

# Fail early: no GPU, or a GPU under 6 GiB, stops here before any data is read.
device = ct.require_cuda(6.0)
print(f"device={torch.cuda.get_device_name(0)} torch={torch.__version__}")
if not OUTPUT_DIR.is_dir():
    raise FileNotFoundError(
        f"{OUTPUT_DIR} does not exist. This notebook is written for Kaggle."
    )
if not ANNOTATIONS_PATH.is_file() or not IMAGE_DIR.is_dir():
    raise FileNotFoundError(
        "Train data was not found. Checked "
        f"images={IMAGE_DIR} and annotations={ANNOTATIONS_PATH}. "
        "Attach the competition filament-segmentation-2026 to this notebook."
    )
low_hours, high_hours = estimated_train_hours(MODEL)
print(f"model={MODEL} expected checkpoint: {EXPECTED.label}")
print(f"estimated training time on a T4: {low_hours:.1f} to {high_hours:.1f} h")
print(CONFIG)
print(f"images={IMAGE_DIR}")
print(f"annotations={ANNOTATIONS_PATH}")
print(f"weights={WEIGHTS_PATH}")
print(f"sweep={SWEEP_PATH}")
print("test/test_images is not used")

## 5. Date holdout and the in-RAM cache

The train JSON is read, the most recent 20% of calendar dates are held out, and each train JPEG is decoded once together with its union mask (with `r34_v2_agree`, the mask holds the agreement weights on the same pixels). This is the slow, CPU-only part before training. A progress line is printed every 50 files.


In [ ]:
data_start = time.monotonic()
loaded = load_train_annotations(ANNOTATIONS_PATH)
split = split_training_images(loaded.images)
holdout_views = count_views(split.holdout)
print(
    f"train files={len(split.train)} holdout files={len(split.holdout)} "
    f"holdout annotator views={holdout_views}"
)
if (len(split.train), len(split.holdout), holdout_views) != (565, 142, 231):
    print(
        "WARNING: the split is not 565 / 142 files with 231 views, the numbers "
        "this run was sized for. CROPS_PER_EPOCH = 4520 is 8 crops per file "
        "only for 565 train files. [VERIFY] the data version."
    )
print(f"oldest holdout file={split.holdout[0].file_name}")

cache = load_train_cache(
    split.train, IMAGE_DIR, agreement_weights=CONFIG.agreement_weights
)
data_prep_seconds = time.monotonic() - data_start
print(
    f"cache={cache.nbytes / 2**30:.2f} GiB "
    f"foreground fraction={cache.foreground_fraction():.4f}"
)
print(f"crops per train file per epoch={CROPS_PER_EPOCH / len(cache.images):.2f}")
print(f"data prep time={data_prep_seconds:.0f} s")

## 6. Train

Dice + BCE, mixed precision, and the optimizer and schedule of `MODEL` (the first log line prints them). The first epoch prints a progress line every 100 steps so the speed can be read early; later epochs print one line each (mean loss, learning rate and seconds; with clipping, also the mean gradient norm and the share of steps it clipped). The checkpoint is rewritten after every epoch with `status` `in_progress`, and once more at the end with `complete`, or `stopped_early` when the wall-clock guard fired. With a weight average, the checkpoint holds the average.


In [ ]:
train_start = time.monotonic()
trained_model, result = ct.train_model(
    cache.images,
    cache.masks,
    CONFIG,
    device,
    checkpoint_path=WEIGHTS_PATH,
    extra_metadata={
        "tile_size": TILE_SIZE,
        "tile_overlap": TILE_OVERLAP,
        "infer_batch_size": INFER_BATCH_SIZE,
        "threshold": THRESHOLD,
    },
)
train_seconds = time.monotonic() - train_start
print(
    f"completed epochs={result.epochs_completed} of {result.epochs_requested} "
    f"stopped_early={result.stopped_early}"
)
if result.stopped_early:
    print(
        "NOTE: the wall-clock guard stopped training early, so the checkpoint "
        f"stores epochs={result.epochs_completed}. The inference notebook "
        f"expects epochs={EXPECTED.epochs}; set its EXPECTED_EPOCHS to "
        f"{result.epochs_completed} (and pin this run's tensor_sha256) to use "
        "these weights."
    )
del trained_model, cache
torch.cuda.empty_cache()

## 7. The checkpoint, read back from disk

The file is loaded again and the network is rebuilt from it (`encoder_weights=None`: the `state_dict` already holds the encoder). The holdout evaluation below uses this rebuilt network, so it scores the artifact that the inference notebook will load.

`tensor_sha256` is the SHA-256 of the `state_dict` tensor bytes, from `filament.checkpoint.tensor_sha256`: the same function the 512 px notebooks use. The 80-epoch run of 2026-11-02 is pinned in `R34_CROPS.tensor_sha256` (the earlier 40-epoch run is `R34_CROPS_40EP`) in `src/filament/checkpoint.py`, so the inference notebook refuses any other file. `R34_V2_AGREE` (D2) is pinned to its run of 2026-11-03. `R50_CROPS`, `EFFB3_CROPS` and `R34_V2` are not pinned yet: after their run, copy the printed value and the `min_area` 400 PQ into that model's entry (`tensor_sha256`, `holdout_pq`), and `epochs` if the run stopped early. The same goes if you retrain `r34`.


In [ ]:
checkpoint = ct.load_checkpoint(WEIGHTS_PATH)
fingerprint = checkpoint_fingerprint(checkpoint)
print(f"checkpoint={WEIGHTS_PATH} bytes={WEIGHTS_PATH.stat().st_size}")
for key in (
    "epochs",
    "epochs_requested",
    "stopped_early",
    "status",
    "encoder_name",
    "encoder_weights",
    "crop_size",
    "training_resolution",
    "native_shape",
    "crops_per_epoch",
    "batch_size",
    "learning_rate",
    "optimizer",
    "weight_decay",
    "lr_schedule",
    "warmup_epochs",
    "grad_clip_norm",
    "ema_decay",
    "p_fg",
    "augment",
    "use_amp",
    "seed",
    "loss",
    "smp_version",
    "tile_size",
    "tile_overlap",
):
    print(f"  {key}={checkpoint.get(key, 'not stored')}")
print(f"tensor_sha256={fingerprint['tensor_sha256']}")
print(
    f"To pin it: set tensor_sha256 of the {MODEL!r} entry in "
    "src/filament/checkpoint.py to this value "
    f"(epochs {fingerprint['epochs']}), and holdout_pq to the min_area 400 row below."
)
eval_model = ct.model_from_checkpoint(checkpoint, device)

## 8. Holdout: tiled inference, then `min_area` sweep

Each of the 142 holdout frames is read at 2048 px and predicted **once** with overlapped tiles (tile 512, overlap 128, batch 8, overlapping probabilities averaged), thresholded at 0.5, and that one mask is reused for every `min_area`. Instances are `filament.instances.labels_from_semantic` at full resolution with `close_size` 0.

The score is the pooled per-view PQ of `notebooks/sweep_postprocess.ipynb`: `filament.metrics.fast_view_counts` per annotator view (`filament.annotations.annotator_views`), `PqCounts` added over all views of all holdout files, one `.pq` at the end. The metric is not reimplemented (`filament.sweep`). The CSV `holdout_sweep_<MODEL>.csv` has the columns `predictor,min_area,close_size,pooled_pq,holdout_files,annotator_views`.

One holdout is one sample of dates. A difference of a few thousandths in PQ between two runs or two `min_area` values is not evidence of a better model.


In [ ]:
sweep = ct.run_holdout_sweep(
    eval_model,
    split.holdout,
    IMAGE_DIR,
    device,
    min_areas=MIN_AREAS,
    out_csv=SWEEP_PATH,
    tile_size=TILE_SIZE,
    overlap=TILE_OVERLAP,
    infer_batch_size=INFER_BATCH_SIZE,
    use_amp=USE_AMP,
    threshold=THRESHOLD,
    close_size=CLOSE_SIZE,
    predictor_name=Path(EXPECTED.weights_path).stem,
)
print(ct.describe_sweep(sweep, SWEEP_PATH))
reference = crop_checkpoint(DEFAULT_CROP_MODEL)
print(
    f"Reference, same holdout, {reference.label}: pooled PQ "
    f"{reference.holdout_pq} at threshold 0.5, min_area 400, close_size 0, no TTA "
    "(docs/RESULTS.md)."
)
print(
    "Reference, same holdout, 512 px ResNet-18 with 12 epochs: pooled PQ 0.2886 "
    "at min_area 400, close_size 0 (docs/RESULTS.md)."
)

## 9. Timings

Data prep is the JSON read plus the JPEG decode and mask rasterization. Epoch times include the optimizer steps and, in epoch 1, the CUDA warm-up. Holdout inference is the time spent in tiled prediction only; the rest of the holdout time is labeling and scoring.


In [ ]:
epoch_times = result.epoch_seconds
print(f"data prep: {data_prep_seconds:.0f} s")
print(
    f"epochs: {len(epoch_times)} | per epoch mean={statistics.fmean(epoch_times):.0f} s "
    f"min={min(epoch_times):.0f} s max={max(epoch_times):.0f} s"
)
print(f"training total: {result.train_seconds / 60:.1f} min")
print(
    f"holdout: inference {sweep.inference_seconds:.0f} s, "
    f"inference + labeling + scoring {sweep.total_seconds:.0f} s"
)
print(
    "notebook total (data prep + training + holdout): "
    f"{(data_prep_seconds + result.train_seconds + sweep.total_seconds) / 3600:.2f} h"
)
print(f"Outputs: {WEIGHTS_PATH.name} and {SWEEP_PATH.name} in /kaggle/working.")
print("This notebook wrote no submission file and did not submit to Kaggle.")